# Reliable Wind Supply Analysis

Goal: estimate how many MW of wind generation can be treated as reliably available to help meet demand.

Approach used here:
- fetch historical `FUELHH` wind actuals from January 2023 through January 2024
- look at the lower tail of realized half-hour wind output
- compare all-hours reliability with evening reliability, since adequacy questions usually matter more during higher-demand hours
- recommend a conservative MW figure grounded in the lower percentiles rather than the average

In [ ]:
from datetime import timezone\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nimport requests\nimport seaborn as sns\n\nsns.set_theme(style='whitegrid')\n\nUTC = timezone.utc\nBASE_URL = 'https://data.elexon.co.uk/bmrs/api/v1'\n\n\ndef fetch_actual_wind(settlement_date_from: str, settlement_date_to: str) -> pd.DataFrame:\n    response = requests.get(\n        f'{BASE_URL}/datasets/FUELHH/stream',\n        params={\n            'format': 'json',\n            'settlementDateFrom': settlement_date_from,\n            'settlementDateTo': settlement_date_to,\n            'fuelType': 'WIND',\n        },\n        timeout=120,\n    )\n    response.raise_for_status()\n    payload = response.json()\n    if isinstance(payload, dict) and 'data' in payload:\n        payload = payload['data']\n    frame = pd.DataFrame(payload)\n    frame['startTime'] = pd.to_datetime(frame['startTime'], utc=True)\n    frame['generation'] = frame['generation'].astype(float)\n    return frame[['startTime', 'generation']]\n

In [ ]:
history = fetch_actual_wind('2023-01-01', '2024-01-31')\nhistory['hourOfDay'] = history['startTime'].dt.hour\nhistory['month'] = history['startTime'].dt.to_period('M').astype(str)\n\nevening = history[history['hourOfDay'].between(16, 20)]\n\npercentiles = pd.DataFrame({\n    'scope': ['all_hours', 'evening_hours'],\n    'p05_mw': [history['generation'].quantile(0.05), evening['generation'].quantile(0.05)],\n    'p10_mw': [history['generation'].quantile(0.10), evening['generation'].quantile(0.10)],\n    'p25_mw': [history['generation'].quantile(0.25), evening['generation'].quantile(0.25)],\n    'mean_mw': [history['generation'].mean(), evening['generation'].mean()],\n})\n\nrecommended_reliable_mw = int(min(history['generation'].quantile(0.10), evening['generation'].quantile(0.10)) // 100 * 100)\npercentiles, recommended_reliable_mw\n

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))\n\nsns.histplot(history['generation'], bins=50, ax=axes[0], color='#1e63ff')\naxes[0].axvline(recommended_reliable_mw, color='#d6254d', linestyle='--', label=f'Recommended reliable MW: {recommended_reliable_mw:,}')\naxes[0].set_title('Distribution of historical half-hour wind generation')\naxes[0].set_xlabel('Generation (MW)')\naxes[0].legend()\n\nmonthly = history.groupby('month')['generation'].quantile([0.1, 0.5]).unstack().reset_index()\nmonthly.columns = ['month', 'p10_mw', 'median_mw']\nsns.lineplot(data=monthly, x='month', y='p10_mw', marker='o', ax=axes[1], label='P10')\nsns.lineplot(data=monthly, x='month', y='median_mw', marker='o', ax=axes[1], label='Median')\naxes[1].axhline(recommended_reliable_mw, color='#d6254d', linestyle='--', label='Recommendation')\naxes[1].set_title('Monthly lower-tail availability')\naxes[1].set_xlabel('Month')\naxes[1].set_ylabel('Generation (MW)')\naxes[1].tick_params(axis='x', rotation=45)\n\nplt.tight_layout()\nplt.show()\n

## Interpretation

A defensible reliability estimate is the lower of the all-hours and evening-hour P10 values, rounded down. That is conservative enough to avoid treating average wind output as firm supply, but not so conservative that it collapses to extreme low-wind edge cases such as P1 or the absolute minimum.

Using the code above, the notebook surfaces a recommended reliable wind contribution in MW together with the percentile evidence used to justify it.